# Strategy: [NOME DA ESTRATÉGIA]

**Função de entrada:** `nome_da_funcao` em `entries/entries.py`  
**Símbolo / Timeframe:** WIN@N / t5  
**Hipótese:** _(descreva a ideia da estratégia)_

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
import importlib
import optuna
from optuna.visualization import plot_param_importances, plot_optimization_history

from futures_backtester import Backtester
from config.dicts_params import dict_custos, dict_valor_lot, dict_path

pd.set_option('display.max_columns', 40)
pd.set_option('display.max_rows', 100)

## 2. Carregar Função de Entrada

In [ ]:
# Altere para o nome da função definida em entries/entries.py
name_strategy = 'nome_da_funcao'

import importlib
module = importlib.import_module('entries')
entrada = getattr(module, name_strategy)
print(f'Estratégia carregada: {name_strategy}')

## 3. Teste Rápido (cenário manual)

In [ ]:
sym = 'WIN@N'

params = {
    'sl': 300,
    'tp': 500,
    # adicione os parâmetros da sua estratégia aqui
}

bt = Backtester(
    symbol=sym,
    timeframe='t5',
    data_ini='2022-01-01',
    data_fim='2025-06-30',
    tp=params['tp'],
    sl=params['sl'],
    slippage=0,
    tc=dict_custos[sym],
    lote=1,
    valor_lote=dict_valor_lot[sym],
    initial_cash=30000,
    path_base=dict_path[sym],
    daytrade=True
)

df, metrics = bt.run(
    signal_function=entrada,
    signal_args={
        # parâmetros da função de entrada
        'allowed_hours': [9, 10, 11, 12, 13],
        'position_type': 'both',
    }
)

bt.print_metrics(metrics)

In [ ]:
bt.plot_equity_curve(include_drawdown=True)
bt.plot_cumulative_by_hour()

## 4. Otimização (Optuna)

In [ ]:
def objective(trial):
    sym = 'WIN@N'

    bt = Backtester(
        symbol=sym,
        timeframe='t5',
        data_ini='2019-01-01',
        data_fim='2025-06-30',
        tp=trial.suggest_int('tp', 100, 800),
        sl=trial.suggest_int('sl', 100, 600),
        slippage=0,
        tc=dict_custos[sym],
        lote=1,
        valor_lote=dict_valor_lot[sym],
        initial_cash=30000,
        path_base=dict_path[sym],
        daytrade=True
    )

    # Adicione os parâmetros da sua estratégia aqui
    _, metrics = bt.run(
        signal_function=entrada,
        signal_args={
            # parâmetros
            'allowed_hours': [9, 10, 11, 12, 13],
            'position_type': 'both',
        }
    )

    return metrics['sharpe_ratio']


study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=200)

print('Melhores parâmetros:', study.best_params)
print('Melhor Sharpe:', study.best_value)

In [ ]:
plot_optimization_history(study)

In [ ]:
plot_param_importances(study)

## 5. Validação com Melhores Parâmetros

In [ ]:
best = study.best_params
print('Rodando com:', best)

bt_val = Backtester(
    symbol=sym,
    timeframe='t5',
    data_ini='2019-01-01',
    data_fim='2025-06-30',
    tp=best['tp'],
    sl=best['sl'],
    slippage=0,
    tc=dict_custos[sym],
    lote=1,
    valor_lote=dict_valor_lot[sym],
    initial_cash=30000,
    path_base=dict_path[sym],
    daytrade=True
)

df_val, metrics_val = bt_val.run(
    signal_function=entrada,
    signal_args={
        # use best params aqui
        'allowed_hours': [9, 10, 11, 12, 13],
        'position_type': 'both',
    }
)

bt_val.print_metrics(metrics_val)
bt_val.plot_equity_curve(include_drawdown=True)